In [0]:
spark.version


'4.1.0'

In [0]:
spark.sql("SHOW SCHEMAS IN fraud_platform").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|feature           |
|gold              |
|information_schema|
|scored            |
|silver            |
|snapshots         |
+------------------+



In [0]:
%sql
CREATE VOLUME IF NOT EXISTS fraud_platform.bronze.streaming_checkpoints;

In [0]:
%sql
SHOW VOLUMES IN fraud_platform.bronze;

database,volume_name
bronze,streaming_checkpoints


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
transaction_schema = StructType([
    StructField("TransactionID", IntegerType()),
    StructField("isFraud", IntegerType()),
    StructField("TransactionDT", IntegerType()),
    StructField("TransactionAmt", DoubleType()),
    StructField("ProductCD", StringType()),
    StructField("card4", StringType()),
    StructField("card6", StringType()),
    StructField("addr1", DoubleType()),
    StructField("addr2", DoubleType()),
    StructField("P_emaildomain", StringType()),
    StructField("R_emaildomain", StringType())
])

In [0]:
# =========================================================================
# Kafka / Confluent Cloud Configuration
# =========================================================================
# These values must be provided as environment variables or Databricks secrets.
# Never hardcode credentials in notebooks.
#
# To set these in Databricks:
#   spark.conf.set("spark.kafka.bootstrap.servers", dbutils.secrets.get(...))
# or use scope-based secrets:
#   API_KEY = dbutils.secrets.get(scope="kafka", key="api_key")
# =========================================================================

import os

BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")

TOPIC = os.getenv("KAFKA_TOPIC", "fraud.transactions.raw")

# Prefer Databricks Secrets over environment variables
try:
    API_KEY = dbutils.secrets.get(scope="kafka", key="api_key")
    API_SECRET = dbutils.secrets.get(scope="kafka", key="api_secret")
except Exception:
    API_KEY = os.getenv("KAFKA_API_KEY", "")
    API_SECRET = os.getenv("KAFKA_API_SECRET", "")

In [0]:
kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", TOPIC)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
    "kafka.sasl.jaas.config",
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'
)
    
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
from pyspark.sql.functions import col, from_json

parsed_df = (
    kafka_df
    .selectExpr("CAST(value AS STRING) AS json")
    .select(
        from_json(
            col("json"),
            transaction_schema
        ).alias("data")
    )
    .select("data.*")
)

In [0]:
query = (
    parsed_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/fraud_platform/bronze/streaming_checkpoints/transactions_raw"
    )
    .toTable("fraud_platform.bronze.transactions_raw")
)

query.awaitTermination()
print("Bronze completed")

Bronze completed


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS fraud_platform;

CREATE SCHEMA IF NOT EXISTS fraud_platform.bronze;

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM fraud_platform.bronze.transactions_raw
""").show()

+--------+
|COUNT(*)|
+--------+
|   20000|
+--------+



In [0]:
print("kafka_df.isStreaming =", kafka_df.isStreaming)

kafka_df.isStreaming = True


In [0]:
print("parsed_df.isStreaming =", parsed_df.isStreaming)

parsed_df.isStreaming = True


In [0]:
%sql
SELECT COUNT(*)
FROM fraud_platform.bronze.transactions_raw;

COUNT(*)
20000


In [0]:
%sql
SELECT *
FROM fraud_platform.bronze.transactions_raw
LIMIT 10;

TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card4,card6,addr1,addr2,P_emaildomain,R_emaildomain
2987000,0,86400,68.5,W,discover,credit,315.0,87.0,null,null
2987017,0,86668,100.0,H,mastercard,credit,204.0,87.0,yahoo.com,null
2987020,0,86761,39.0,W,mastercard,debit,299.0,87.0,gmail.com,null
2987032,0,87008,200.0,W,visa,debit,472.0,87.0,yahoo.com,null
2987048,0,87317,42.294,C,visa,debit,null,null,outlook.com,outlook.com
2987049,0,87317,3.595,C,mastercard,credit,null,null,anonymous.com,anonymous.com
2987050,0,87319,117.0,W,visa,debit,299.0,87.0,gmail.com,null
2987062,0,87601,200.0,W,mastercard,debit,204.0,87.0,yahoo.com,null
2987064,0,87611,250.0,W,visa,debit,122.0,87.0,gmail.com,null
2987070,0,87735,100.0,H,visa,credit,325.0,87.0,anonymous.com,null


In [0]:
%sql
DESCRIBE DETAIL fraud_platform.bronze.transactions_raw;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,5cdbd055-0353-4aae-8ceb-0d6951597cb8,fraud_platform.bronze.transactions_raw,null,,2026-07-04T11:31:43.255Z,2026-07-06T08:15:04.000Z,List(),List(),6,137496,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-c4d379e4-7f2a-42e2-a496-cbb78c580f23, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-745f53e1-bee8-4950-9567-42e784e0b4cf)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
